In [ ]:
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister
from qiskit.quantum_info import *
from qiskit.visualization import plot_histogram
from IPython.display import display
from qiskit.circuit.library import UnitaryGate
from numpy import pi
from qiskit_aer import AerSimulator
import numpy as np
import math
import random

# Cost circuit
def make_circuit(theta1,theta2):
    qc = QuantumCircuit(2)
    # The choice of Ansatz depends...basically here we have taken a 
    # Hamiltonian that couples |10> with |01> so we have taken a cx also..
    
    qc.ry(theta1, 0)

    return qc


# Cost function
# We take cost = <Z0>

def cost(theta1, theta2):
    qc = make_circuit(theta1, theta2)

    state = Statevector.from_instruction(qc)

    X = Operator([[0, 1],
                  [1, 0]])

    Z = Operator([[1, 0],
                  [0, -1]])

    I = Operator([[1, 0],
                  [0, 1]])

    H = I.tensor(Z)

    new_state = state.evolve(H)

    value = state.inner(new_state)

    return np.real(value)

# Parameter Shift Rule
def psr_gradient(theta1, theta2):

    # dC/dtheta1
    grad1 = (
        cost(theta1 + pi/2, theta2)
        - cost(theta1 - pi/2, theta2)
    ) / 2

    # dC/dtheta2
    grad2 = (
        cost(theta1, theta2 + pi/2)
        - cost(theta1, theta2 - pi/2)
    ) / 2

    return grad1, grad2


# Gradient Descent

theta1 = random.uniform(0, 2*pi)
theta2 = random.uniform(0, 2*pi)

eta = 0.2

max_iter = 100
tol = 1e-6

print("Initial theta1 =", theta1)
print("Initial theta2 =", theta2)
print("Initial cost   =", cost(theta1, theta2))

for step in range(max_iter):

    grad1, grad2 = psr_gradient(theta1, theta2)

    theta1_new = theta1 - eta*grad1
    theta2_new = theta2 - eta*grad2

    c_old = cost(theta1, theta2)
    c_new = cost(theta1_new, theta2_new)

    print(
        f"Step {step:3d} | "
        f"Cost = {c_old:.6f} | "
        f"Theta1 = {theta1:.6f} | "
        f"Theta2 = {theta2:.6f} | "
        f"Grad1 = {grad1:.6f} | "
        f"Grad2 = {grad2:.6f}"
    )

    # Better convergence criterion
    if abs(grad1) < tol and abs(grad2) < tol:
        print("\nConverged.")
        theta1 = theta1_new
        theta2 = theta2_new
        break

    theta1 = theta1_new
    theta2 = theta2_new


print("\nFinal theta1 =", theta1)
print("Final theta2 =", theta2)
print("Final cost   =", cost(theta1, theta2))

Initial theta1 = 0.9717402078637053
Initial theta2 = 2.6112111289877533
Initial cost   = 0.5638632033251714
Step   0 | Cost = 0.563863 | Theta1 = 0.971740 | Theta2 = 2.611211 | Grad1 = -0.825868 | Grad2 = 0.000000
Step   1 | Cost = 0.420397 | Theta1 = 1.136914 | Theta2 = 2.611211 | Grad1 = -0.907340 | Grad2 = 0.000000
Step   2 | Cost = 0.249743 | Theta1 = 1.318382 | Theta2 = 2.611211 | Grad1 = -0.968312 | Grad2 = 0.000000
Step   3 | Cost = 0.058718 | Theta1 = 1.512044 | Theta2 = 2.611211 | Grad1 = -0.998275 | Grad2 = 0.000000
Step   4 | Cost = -0.140437 | Theta1 = 1.711699 | Theta2 = 2.611211 | Grad1 = -0.990090 | Grad2 = 0.000000
Step   5 | Cost = -0.332470 | Theta1 = 1.909717 | Theta2 = 2.611211 | Grad1 = -0.943114 | Grad2 = 0.000000
Step   6 | Cost = -0.503412 | Theta1 = 2.098340 | Theta2 = 2.611211 | Grad1 = -0.864046 | Grad2 = 0.000000
Step   7 | Cost = -0.644488 | Theta1 = 2.271149 | Theta2 = 2.611211 | Grad1 = -0.764615 | Grad2 = 0.000000
Step   8 | Cost = -0.753438 | Theta1 = 2